# 07 — Validações obrigatórias, dicionário de dados e relatório

**Objetivo:** fechar a pipeline garantindo que as regras críticas foram respeitadas (seção 9 da proposta). Cada verificação é um `assert`: se alguma regra for violada, **este notebook falha** em vez de exportar dados errados.

Depois das validações:
- consolida `qualidade_dados.csv` (inventário do 01 + descartes do 02 + resultado das validações);
- gera `dicionario_dados.csv` (colunas e granularidade de cada CSV exportado);
- escreve `relatorio_analitico.md` com os números **calculados** nos notebooks anteriores.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import re
import pandas as pd
import numpy as np
import olist_utils as U

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.max_rows", 200)

pedidos = U.load_processed("pedidos")
itens = U.load_processed("itens")
ps = U.load_processed("pedido_seller")
pc = U.load_processed("pedido_categoria")
raw_orders = U.load_raw("orders")
raw_items = U.load_raw("items")
raw_reviews = U.load_raw("reviews")

validacoes = []
def validar(nome, condicao, detalhe=""):
    ok = bool(condicao)
    validacoes.append({"etapa": "07_validacao", "tabela": "pipeline", "verificacao": nome,
                       "valor": int(ok), "detalhe": ("OK " if ok else "FALHOU ") + detalhe})
    print(("✔" if ok else "✘"), nome, detalhe)
    assert ok, nome

## 1. Granularidade e joins

In [2]:
validar("pedidos: 1 linha por order_id", pedidos["order_id"].is_unique, f"{len(pedidos)} linhas")
validar("itens: 1 linha por order_id+order_item_id", not itens.duplicated(["order_id", "order_item_id"]).any())
validar("pedido_seller: 1 linha por order_id+seller_id", not ps.duplicated(["order_id", "seller_id"]).any())
validar("pedido_seller identificado como tal (linhas > pedidos únicos)",
        (ps["granularidade"] == "pedido_seller").all() and len(ps) > ps["order_id"].nunique(),
        f"{len(ps)} linhas para {ps['order_id'].nunique()} pedidos únicos")

ids = set(pedidos["order_id"])
ri = raw_items[raw_items["order_id"].isin(ids)]
validar("itens: mesma quantidade que os itens originais dos pedidos do recorte", len(itens) == len(ri))
validar("frete não multiplicado pelos joins (itens)", np.isclose(itens["freight_value"].sum(), ri["freight_value"].sum()),
        f"{itens['freight_value'].sum():.2f}")
validar("preço não multiplicado pelos joins (itens)", np.isclose(itens["price"].sum(), ri["price"].sum()))
validar("frete pedido_seller == frete itens", np.isclose(ps["soma_frete"].sum(), itens["freight_value"].sum()))
validar("frete pedidos == frete itens", np.isclose(pedidos["soma_frete"].sum(), itens["freight_value"].sum()))

✔ pedidos: 1 linha por order_id 74781 linhas
✔ itens: 1 linha por order_id+order_item_id 
✔ pedido_seller: 1 linha por order_id+seller_id 


✔ pedido_seller identificado como tal (linhas > pedidos únicos) 75845 linhas para 74781 pedidos únicos
✔ itens: mesma quantidade que os itens originais dos pedidos do recorte 
✔ frete não multiplicado pelos joins (itens) 1712901.77
✔ preço não multiplicado pelos joins (itens) 
✔ frete pedido_seller == frete itens 
✔ frete pedidos == frete itens 


## 2. Filtros de período e status

In [3]:
ro = raw_orders.set_index("order_id").loc[pedidos["order_id"]]
validar("somente status delivered", (ro["order_status"] == "delivered").all())
validar("ano_original confere com a data de compra original",
        (ro["order_purchase_timestamp"].dt.year.values == pedidos["ano_original"].values).all())
validar("somente meses 1 a 8", pedidos["mes"].between(1, 8).all())
validar("somente anos 2017 e 2018", set(pedidos["ano_original"]) == {2017, 2018})
validar("datas originais não alteradas",
        (ro["order_purchase_timestamp"].values == pedidos["order_purchase_timestamp"].values).all())
validar("rótulo de período coerente com o ano original",
        (pedidos["periodo_comparativo"] == pedidos["ano_original"].map(U.PERIODOS)).all())
n_ano = pedidos.groupby("ano_original").size()
for ano, ref in U.CHECK_PEDIDOS_PROPOSTA.items():
    # registrado, mas não forçado: a validação só documenta a comparação
    validacoes.append({"etapa": "07_validacao", "tabela": "pipeline",
                       "verificacao": f"pedidos {ano} vs referência da proposta", "valor": int(n_ano[ano]),
                       "detalhe": f"referência {ref}; diferença {int(n_ano[ano]) - ref}"})
    print(f"  pedidos {ano}: {n_ano[ano]} (referência {ref})")

✔ somente status delivered 
✔ ano_original confere com a data de compra original 
✔ somente meses 1 a 8 
✔ somente anos 2017 e 2018 
✔ datas originais não alteradas 
✔ rótulo de período coerente com o ano original 
  pedidos 2017: 21998 (referência 21998)
  pedidos 2018: 52783 (referência 52783)


## 3. Avaliação mais recente por pedido
Recalculamos de forma independente (sem usar a função do projeto): para cada pedido com mais de uma avaliação, a data escolhida deve ser a maior `review_creation_date` disponível.

In [4]:
rr = raw_reviews[raw_reviews["order_id"].isin(ids)]
max_data = rr.groupby("order_id")["review_creation_date"].max()
multi = rr.groupby("order_id").size()
multi = multi[multi > 1].index
escolhida = pedidos.set_index("order_id").loc[multi, "review_creation_date"]
validar("avaliação escolhida é a mais recente", (escolhida == max_data.loc[multi]).all(), f"{len(multi)} pedidos com >1 avaliação")
validar("notas no intervalo 1-5", pedidos["review_score"].dropna().between(1, 5).all())
validar("avaliacao_ruim == nota 1 ou 2",
        (pedidos.loc[pedidos["tem_avaliacao"], "avaliacao_ruim"].astype(bool)
         == pedidos.loc[pedidos["tem_avaliacao"], "review_score"].isin([1, 2])).all())
validar("pedidos sem avaliação não têm nota imputada", pedidos.loc[~pedidos["tem_avaliacao"], "review_score"].isna().all())

✔ avaliação escolhida é a mais recente 391 pedidos com >1 avaliação
✔ notas no intervalo 1-5 
✔ avaliacao_ruim == nota 1 ou 2 
✔ pedidos sem avaliação não têm nota imputada 


## 4. Regra do prazo de postagem

In [5]:
min_lim = ri.groupby("order_id")["shipping_limit_date"].min()
validar("KPI3 por pedido usa o menor shipping_limit_date",
        (pedidos.set_index("order_id")["prazo_postagem_limite"] == min_lim.reindex(pedidos["order_id"]).values).all())
e = pedidos[pedidos["postagem_elegivel"]]
validar("fronteira: postagem == limite conta como no prazo",
        (e["postagem_no_prazo"].astype(bool) == (e["order_delivered_carrier_date"] <= e["prazo_postagem_limite"])).all())
validar("não elegíveis ficam sem classificação", pedidos.loc[~pedidos["postagem_elegivel"], "postagem_no_prazo"].isna().all())
min_ps = ri.groupby(["order_id", "seller_id"])["shipping_limit_date"].min()
validar("pedido_seller usa o menor shipping_limit_date do seller no pedido",
        (ps.set_index(["order_id", "seller_id"])["prazo_postagem_limite_seller"] == min_ps.reindex(
            pd.MultiIndex.from_frame(ps[["order_id", "seller_id"]])).values).all())

✔ KPI3 por pedido usa o menor shipping_limit_date 
✔ fronteira: postagem == limite conta como no prazo 
✔ não elegíveis ficam sem classificação 


✔ pedido_seller usa o menor shipping_limit_date do seller no pedido 


## 5. Mapa UF → região

In [6]:
validar("mapa UF->região cobre as UFs presentes", pedidos["regiao_cliente"].notna().all(),
        f"{pedidos['customer_state'].nunique()} UFs")
validar("mapa tem 27 UFs e 5 regiões", len(U.UF_REGIAO) == 27 and len(set(U.UF_REGIAO.values())) == 5)

✔ mapa UF->região cobre as UFs presentes 27 UFs
✔ mapa tem 27 UFs e 5 regiões 


## 6. CSVs exportados: releitura, numeradores ≤ denominadores e percentuais em 0–100
Exceção documentada: o KPI2 (frete/preço) é uma razão e pode passar de 100% em grupos pequenos (ex.: seller que vende itens baratos com frete caro); para ele exige-se apenas ≥ 0.

In [7]:
PARES = [  # (numerador, denominador) — verificados em qualquer CSV que tenha as duas colunas
    ("pedidos_avaliacao_ruim", "pedidos_com_avaliacao"), ("pedidos_com_avaliacao", "pedidos_entregues"),
    ("pedidos_postagem_no_prazo", "pedidos_postagem_elegiveis"), ("pedidos_postagem_elegiveis", "pedidos_entregues"),
    ("pedidos_entrega_atrasada", "pedidos_entrega_avaliavel"),
    ("pedido_seller_postagem_atrasada", "pedido_seller_elegiveis_postagem"),
    ("pedido_seller_avaliacao_ruim", "pedido_seller_com_avaliacao"),
    ("pedido_seller_entrega_atrasada", "pedido_seller_entrega_avaliavel"),
    ("pedido_seller_atrasadas", "pedido_seller_elegiveis"), ("numerador", "denominador"),
]
EXCLUI_PCT = re.compile(r"diferenca|referencia|meta|razao|variacao")
csvs = {}
for f in sorted(U.DASHBOARD_DIR.glob("*.csv")):
    df = pd.read_csv(f)
    csvs[f.stem] = df
    for a, b in PARES:
        if a in df and b in df:
            m = df[[a, b]].dropna()
            validar(f"{f.name}: {a} <= {b}", (m[a] <= m[b]).all())
    pcts = [c for c in df.columns if (c.startswith("pct") or re.match(r"kpi\d_pct", c)) and not EXCLUI_PCT.search(c)]
    for c in pcts:
        if "frete" in c:
            # frete/preço é uma razão: pode passar de 100% (item barato com frete caro). Exige-se apenas >= 0.
            acima = int((df[c] > 100).sum())
            validar(f"{f.name}: {c} >= 0", (df[c].dropna() >= 0).all(), f"{acima} grupos acima de 100% (razão legítima)")
        else:
            validar(f"{f.name}: {c} entre 0 e 100", df[c].dropna().between(0, 100).all())
print(len(csvs), "CSVs relidos sem erro")

✔

 distribuicao_notas.csv: pct_pedidos_avaliados entre 0 e 100 
✔ kpi_categoria.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ kpi_categoria.csv: pedidos_com_avaliacao <= pedidos_entregues 
✔ kpi_categoria.csv: pedidos_postagem_no_prazo <= pedidos_postagem_elegiveis 
✔ kpi_categoria.csv: pedidos_postagem_elegiveis <= pedidos_entregues 
✔ kpi_categoria.csv: pedidos_entrega_atrasada <= pedidos_entrega_avaliavel 
✔ kpi_categoria.csv: pct_sem_avaliacao entre 0 e 100 
✔ kpi_categoria.csv: kpi1_pct_avaliacao_ruim entre 0 e 100 
✔ kpi_categoria.csv: kpi3_pct_postagem_no_prazo entre 0 e 100 
✔ kpi_categoria.csv: pct_entrega_atrasada entre 0 e 100 
✔ kpi_categoria.csv: kpi2_pct_frete_sobre_produto >= 0 0 grupos acima de 100% (razão legítima)
✔ kpi_mensal.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ kpi_mensal.csv: pedidos_com_avaliacao <= pedidos_entregues 
✔ kpi_mensal.csv: pedidos_postagem_no_prazo <= pedidos_postagem_elegiveis 
✔ kpi_mensal.csv: pedidos_postagem_elegiveis <

✔ kpi_regiao.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ kpi_regiao.csv: pedidos_com_avaliacao <= pedidos_entregues 
✔ kpi_regiao.csv: pedidos_postagem_no_prazo <= pedidos_postagem_elegiveis 
✔ kpi_regiao.csv: pedidos_postagem_elegiveis <= pedidos_entregues 
✔ kpi_regiao.csv: pedidos_entrega_atrasada <= pedidos_entrega_avaliavel 
✔ kpi_regiao.csv: pct_sem_avaliacao entre 0 e 100 
✔ kpi_regiao.csv: kpi1_pct_avaliacao_ruim entre 0 e 100 
✔ kpi_regiao.csv: kpi3_pct_postagem_no_prazo entre 0 e 100 
✔ kpi_regiao.csv: pct_entrega_atrasada entre 0 e 100 
✔ kpi_regiao.csv: kpi2_pct_frete_sobre_produto >= 0 0 grupos acima de 100% (razão legítima)
✔ kpi_resumo_periodo.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ kpi_resumo_periodo.csv: pedidos_com_avaliacao <= pedidos_entregues 
✔ kpi_resumo_periodo.csv: pedidos_postagem_no_prazo <= pedidos_postagem_elegiveis 
✔ kpi_resumo_periodo.csv: pedidos_postagem_elegiveis <= pedidos_entregues 
✔ kpi_resumo_periodo.csv: pedidos_entre

✔ kpi_seller.csv: pedido_seller_postagem_atrasada <= pedido_seller_elegiveis_postagem 


✔ kpi_seller.csv: pedido_seller_avaliacao_ruim <= pedido_seller_com_avaliacao 
✔ kpi_seller.csv: pedido_seller_entrega_atrasada <= pedido_seller_entrega_avaliavel 
✔ kpi_seller.csv: pct_postagem_atrasada entre 0 e 100 
✔ kpi_seller.csv: pct_postagem_no_prazo entre 0 e 100 
✔ kpi_seller.csv: pct_avaliacao_ruim entre 0 e 100 
✔ kpi_seller.csv: pct_entrega_atrasada entre 0 e 100 
✔ kpi_seller.csv: pct_frete_sobre_produto >= 0 91 grupos acima de 100% (razão legítima)
✔ kpi_uf.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ kpi_uf.csv: pedidos_com_avaliacao <= pedidos_entregues 
✔ kpi_uf.csv: pedidos_postagem_no_prazo <= pedidos_postagem_elegiveis 
✔ kpi_uf.csv: pedidos_postagem_elegiveis <= pedidos_entregues 
✔ kpi_uf.csv: pedidos_entrega_atrasada <= pedidos_entrega_avaliavel 
✔ kpi_uf.csv: pct_sem_avaliacao entre 0 e 100 
✔ kpi_uf.csv: kpi1_pct_avaliacao_ruim entre 0 e 100 
✔ kpi_uf.csv: kpi3_pct_postagem_no_prazo entre 0 e 100 
✔ kpi_uf.csv: pct_entrega_atrasada entre 0 e 100 
✔ 

✔ ranking_semanal_sellers_atraso.csv: pedido_seller_postagem_atrasada <= pedido_seller_elegiveis_postagem 
✔ ranking_semanal_sellers_atraso.csv: pct_postagem_atrasada entre 0 e 100 
✔ relacao_atraso_avaliacao.csv: pedidos_avaliacao_ruim <= pedidos_com_avaliacao 
✔ relacao_atraso_avaliacao.csv: pedidos_entrega_atrasada <= pedidos_entrega_avaliavel 
✔ relacao_atraso_avaliacao.csv: pct_avaliacao_ruim entre 0 e 100 
✔ relacao_atraso_avaliacao.csv: pct_entrega_atrasada entre 0 e 100 
✔ testes_associacao.csv: numerador <= denominador 
✔ testes_associacao.csv: pct entre 0 e 100 
✔ top_50_sellers_atraso.csv: pedido_seller_postagem_atrasada <= pedido_seller_elegiveis_postagem 
✔ top_50_sellers_atraso.csv: pedido_seller_avaliacao_ruim <= pedido_seller_com_avaliacao 
✔ top_50_sellers_atraso.csv: pedido_seller_entrega_atrasada <= pedido_seller_entrega_avaliavel 
✔ top_50_sellers_atraso.csv: pct_postagem_atrasada entre 0 e 100 
✔ top_50_sellers_atraso.csv: pct_postagem_no_prazo entre 0 e 100 
✔ top

In [8]:
resumo = csvs["kpi_resumo_periodo"]
validar("KPI1 do resumo recalculado a partir da tabela de pedidos",
        np.allclose(resumo.sort_values("ano_original")["kpi1_pct_avaliacao_ruim"],
                    pedidos[pedidos.tem_avaliacao].groupby("ano_original")["avaliacao_ruim"].mean().astype(float) * 100))
validar("KPI2 do resumo recalculado a partir dos itens originais",
        np.allclose(resumo.sort_values("ano_original")["kpi2_pct_frete_sobre_produto"],
                    ri.merge(pedidos[["order_id", "ano_original"]]).groupby("ano_original")
                      .apply(lambda d: d.freight_value.sum() / d.price.sum() * 100, include_groups=False)))
mensal = csvs["kpi_mensal"]
validar("soma mensal de pedidos == total do período",
        (mensal.groupby("ano_original")["pedidos_entregues"].sum().values ==
         resumo.sort_values("ano_original")["pedidos_entregues"].values).all())
reg = csvs["kpi_regiao"]
validar("soma por região de pedidos == total do período",
        (reg.groupby("ano_original")["pedidos_entregues"].sum().values ==
         resumo.sort_values("ano_original")["pedidos_entregues"].values).all())

✔ KPI1 do resumo recalculado a partir da tabela de pedidos 


✔ KPI2 do resumo recalculado a partir dos itens originais 
✔ soma mensal de pedidos == total do período 
✔ soma por região de pedidos == total do período 


## 7. `qualidade_dados.csv`

In [9]:
qualidade = pd.concat([U.load_processed("qualidade_01"), U.load_processed("qualidade_02"),
                       pd.DataFrame(validacoes)], ignore_index=True)
qualidade["granularidade"] = "verificacao"
U.export_csv(qualidade, "qualidade_dados")
print(len(qualidade), "linhas em qualidade_dados.csv;", int((qualidade.etapa == "07_validacao").sum()), "validações")

216 linhas em qualidade_dados.csv; 113 validações


## 8. `dicionario_dados.csv`
Descrição de cada coluna de cada CSV exportado. Colunas com padrão conhecido recebem a descrição por regra; a granularidade vem da coluna `granularidade` de cada arquivo.

In [10]:
DESC = {
    "ano_original": "Ano real da compra no dataset (2017 ou 2018). Datas originais não são alteradas.",
    "periodo_comparativo": "Rótulo da proxy acadêmica: 2024 = jan-ago/2017; 2025 = jan-ago/2018. Não são dados reais de 2024/2025.",
    "mes": "Mês da compra (1-8).", "ano_mes_original": "Ano-mês original da compra (AAAA-MM).",
    "regiao_cliente": "Região brasileira derivada da UF do cliente (mapa explícito de 27 UFs).",
    "customer_state": "UF do cliente.", "seller_state": "UF do seller.", "seller_regiao": "Região derivada da UF do seller.",
    "seller_id": "Identificador do seller.", "categoria_en": "Categoria do produto em inglês (tradução oficial; nome original se sem tradução; 'sem_categoria' se ausente).",
    "pedidos_entregues": "Pedidos únicos entregues (status delivered) no grupo.",
    "pedidos_com_avaliacao": "Denominador do KPI1: pedidos com avaliação válida (mais recente por pedido).",
    "pedidos_avaliacao_ruim": "Numerador do KPI1: pedidos cuja avaliação mais recente tem nota 1 ou 2.",
    "pedidos_sem_avaliacao": "Pedidos entregues sem avaliação (fora do denominador do KPI1).",
    "pct_sem_avaliacao": "Pedidos sem avaliação / pedidos entregues x 100.",
    "pedidos_postagem_elegiveis": "Denominador do KPI3: pedidos com data de postagem e prazo limite válidos.",
    "pedidos_postagem_no_prazo": "Numerador do KPI3: pedidos postados até o menor shipping_limit_date dos itens (<= limite).",
    "pedidos_entrega_avaliavel": "Pedidos com data de entrega real e estimada válidas.",
    "pedidos_entrega_atrasada": "Pedidos entregues ao cliente depois do dia estimado.",
    "pct_entrega_atrasada": "Entrega atrasada / entregas avaliáveis x 100.",
    "kpi1_pct_avaliacao_ruim": "KPI1: pedidos com nota 1-2 / pedidos com avaliação x 100.",
    "kpi2_pct_frete_sobre_produto": "KPI2: soma do frete dos itens / soma do preço dos itens x 100.",
    "kpi3_pct_postagem_no_prazo": "KPI3: pedidos postados no prazo / pedidos elegíveis x 100.",
    "itens": "Quantidade de itens (linhas de order_items).", "soma_frete": "Numerador do KPI2: soma de freight_value dos itens (R$).",
    "soma_preco_produtos": "Denominador do KPI2: soma de price dos itens (R$).", "soma_preco": "Soma de price dos itens (R$).",
    "amostra_pequena": f"True quando o denominador principal do grupo é menor que {U.MIN_AMOSTRA}; interpretar com cautela.",
    "granularidade": "Unidade de análise de cada linha do arquivo.",
    "rank": "Posição no ranking por número de postagens atrasadas.", "rank_semana": "Posição no ranking da semana.",
    "semana_prazo_postagem": "Segunda-feira da semana do prazo limite de postagem.",
    "situacao_postagem": "postagem no prazo / postagem atrasada / todos / sem dado.",
    "situacao_entrega": "entrega no prazo / entrega atrasada / todos / sem dado.",
    "numerador": "Numerador da taxa do grupo.", "denominador": "Denominador da taxa do grupo.", "pct": "numerador / denominador x 100.",
    "qui2": "Estatística qui-quadrado (associação, não causalidade).", "p_valor": "p-valor do qui-quadrado.",
    "comparacao": "Qual comparação o teste avalia.", "grupo": "Grupo comparado.",
    "verificacao": "Nome da verificação de qualidade/validação.", "valor": "Resultado numérico (para validações: 1 = OK).",
    "detalhe": "Detalhe/observação.", "etapa": "Notebook que gerou a verificação.", "tabela": "Tabela de origem.",
    "review_score": "Nota da avaliação (1-5).", "pedidos_referencia_proposta": "Total de pedidos informado na proposta (check, não calculado).",
}
def descrever(col):
    if col in DESC: return DESC[col]
    if col.endswith("_referencia_proposta"): return "Valor de referência informado na proposta (não calculado)."
    if "_diferenca_pp_vs_referencia" in col: return "Calculado (1 casa) menos referência da proposta, em p.p."
    if col.endswith("_meta_2026") or col.endswith("_meta_2027"): return "Meta da proposta (não calculada)."
    if col.startswith("pedido_seller_"): return "Contagem na granularidade pedido-seller (não equivale a pedidos únicos): " + col.replace("pedido_seller_", "").replace("_", " ") + "."
    if col.startswith("pct_"): return "Percentual (0-100): " + col[4:].replace("_", " ") + "."
    if col.startswith("mediana_"): return "Mediana: " + col[8:].replace("_", " ") + "."
    return col.replace("_", " ").capitalize() + "."

dic = []
for nome, df in csvs.items():
    if nome in ("dicionario_dados",): continue
    gran = "; ".join(map(str, df["granularidade"].dropna().unique())) if "granularidade" in df else ""
    for c in df.columns:
        dic.append({"arquivo": f"{nome}.csv", "linhas_arquivo": len(df), "granularidade_arquivo": gran,
                    "coluna": c, "tipo": str(df[c].dtype), "descricao": descrever(c)})
dic.append({"arquivo": "qualidade_dados.csv", "linhas_arquivo": len(qualidade), "granularidade_arquivo": "verificacao",
            "coluna": "(arquivo)", "tipo": "", "descricao": "Inventário, descartes e validações da pipeline."})
dicionario = pd.DataFrame(dic)
U.export_csv(dicionario, "dicionario_dados")
dicionario.groupby("arquivo").agg(colunas=("coluna", "size"), linhas=("linhas_arquivo", "first"))

,colunas,linhas
arquivo,,
distribuicao_notas.csv,4,10
kpi_categoria.csv,21,144
kpi_mensal.csv,21,16
kpi_regiao.csv,22,10
kpi_resumo_periodo.csv,32,2
kpi_seller.csv,23,6293
kpi_uf.csv,22,54
kpi_uf_seller_postagem.csv,8,40
origem_atraso_regiao.csv,10,10


## 9. Relatório analítico (`relatorio_analitico.md`)
Montado a partir dos CSVs exportados e das tabelas salvas: nenhum número é digitado à mão.

In [11]:
r = resumo.set_index("ano_original")
P24, P25 = U.PERIODOS[2017], U.PERIODOS[2018]
f1 = lambda x: f"{x:.1f}".replace(".", ",")
f2 = lambda x: f"{x:.2f}".replace(".", ",")
n0 = lambda x: f"{int(round(x)):,}".replace(",", ".")

testes = csvs["testes_associacao"]
def t(comp, per, grupo):
    return testes[(testes.comparacao == comp) & (testes.periodo_comparativo == per) & (testes.grupo == grupo)].iloc[0]
ent_atr = t("avaliacao_ruim_por_situacao_entrega", P25, "entrega atrasada")
ent_ok = t("avaliacao_ruim_por_situacao_entrega", P25, "entrega no prazo")
pos_atr = t("avaliacao_ruim_por_situacao_postagem", P25, "postagem atrasada")
pos_ok = t("avaliacao_ruim_por_situacao_postagem", P25, "postagem no prazo")
pe_atr = t("entrega_atrasada_por_situacao_postagem", P25, "postagem atrasada")
pe_ok = t("entrega_atrasada_por_situacao_postagem", P25, "postagem no prazo")
ent_atr24 = t("avaliacao_ruim_por_situacao_entrega", P24, "entrega atrasada")
ent_ok24 = t("avaliacao_ruim_por_situacao_entrega", P24, "entrega no prazo")

rel = csvs["relacao_atraso_avaliacao"]
def c(per, p, e, col="pct_avaliacao_ruim"):
    return rel[(rel.periodo_comparativo == per) & (rel.situacao_postagem == p) & (rel.situacao_entrega == e)][col].iloc[0]
atr_post_ok = c(P25, "postagem no prazo", "entrega atrasada", "pedidos_com_avaliacao")
atr_total = c(P25, "todos", "entrega atrasada", "pedidos_com_avaliacao")
share_log = atr_post_ok / atr_total * 100

dec = U.load_processed("decomposicao_entrega").set_index("periodo_comparativo")
decp = U.load_processed("decomposicao_postagem").set_index("periodo_comparativo")
orig = csvs["origem_atraso_regiao"]; orig25 = orig[orig.periodo_comparativo == P25].set_index("regiao_cliente")
reg25 = reg[reg.ano_original == 2018].set_index("regiao_cliente")
conc = U.load_processed("concentracao_atrasos").set_index("top_n_sellers")
uf = csvs["kpi_uf"]; uf25 = uf[uf.ano_original == 2018].copy()
taxa = uf25.pedidos_avaliacao_ruim.sum() / uf25.pedidos_com_avaliacao.sum()
uf25["excesso"] = uf25.pedidos_avaliacao_ruim - uf25.pedidos_com_avaliacao * taxa
top_uf = uf25.sort_values("excesso", ascending=False).head(5)
cat = csvs["kpi_categoria"]; cat25 = cat[(cat.ano_original == 2018) & (~cat.amostra_pequena)].copy()
cat25["excesso"] = cat25.pedidos_avaliacao_ruim - cat25.pedidos_com_avaliacao * taxa
top_cat = cat25.sort_values("excesso", ascending=False).head(6)
low_post_cat = cat25[cat25.pedidos_entregues >= 200].sort_values("kpi3_pct_postagem_no_prazo").head(5)
mens = mensal[mensal.ano_original == 2018].set_index("mes")
fr_ent = U.kpi_frete(itens[(itens.ano_original == 2018) & itens.entrega_atrasada.notna()].assign(
    ea=lambda d: d.entrega_atrasada.astype(bool)), "ea").set_index("ea")["kpi2_pct_frete_sobre_produto"]
bate_ref = all(int(n_ano[a]) == ref for a, ref in U.CHECK_PEDIDOS_PROPOSTA.items())
top50 = csvs["top_50_sellers_atraso"]; top50c = top50[top50.periodo_comparativo.str.startswith("consolidado")]

def tabela(df, cols, nomes, fmt):
    linhas = ["| " + " | ".join(nomes) + " |", "|" + "|".join(["---"] + ["---:"] * (len(nomes) - 1)) + "|"]
    for idx, row in df.iterrows():
        linhas.append("| " + " | ".join([str(idx)] + [fmt[i](row[col]) for i, col in enumerate(cols)]) + " |")
    return "\n".join(linhas)

kpi_tab = []
for kpi, nome in [("kpi1_pct_avaliacao_ruim", "KPI1 — % avaliação ruim (1–2)"),
                  ("kpi2_pct_frete_sobre_produto", "KPI2 — % frete / valor dos produtos"),
                  ("kpi3_pct_postagem_no_prazo", "KPI3 — % postagem no prazo")]:
    ref = U.REFERENCIAS[kpi]
    kpi_tab.append(f"| {nome} | {f2(r.loc[2017, kpi])} | {f1(ref['ref_2024'])} | {f2(r.loc[2018, kpi])} | "
                   f"{f1(ref['ref_2025'])} | {f1(ref['meta_2026'])} | {f1(ref['meta_2027'])} |")
kpi_tab = "\n".join(kpi_tab)

reg_tab = tabela(reg25.loc[["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]],
                 ["pedidos_entregues", "kpi1_pct_avaliacao_ruim", "pct_entrega_atrasada", "kpi2_pct_frete_sobre_produto",
                  "kpi2_referencia_proposta_2025", "kpi3_pct_postagem_no_prazo"],
                 ["Região (2025 proxy)", "Pedidos", "% aval. ruim", "% entrega atrasada", "% frete/produto",
                  "ref. frete proposta", "% postagem no prazo"],
                 [n0, f2, f2, f2, lambda x: "—" if pd.isna(x) else f1(x), f2])
orig_tab = tabela(orig25.loc[["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]],
                  ["pct_postagem_atrasada", "pct_entrega_atrasada_dado_postagem_no_prazo", "mediana_dias_postagem_entrega"],
                  ["Região (2025 proxy)", "% postagem atrasada", "% entrega atrasada c/ postagem no prazo",
                   "mediana dias postagem→entrega"], [f2, f2, f1])
uf_tab = tabela(top_uf.set_index("customer_state"),
                ["regiao_cliente", "pedidos_com_avaliacao", "kpi1_pct_avaliacao_ruim", "pct_entrega_atrasada", "kpi2_pct_frete_sobre_produto", "excesso"],
                ["UF", "Região", "Pedidos avaliados", "% aval. ruim", "% entrega atrasada", "% frete", "pedidos ruins acima da média"],
                [str, n0, f2, f2, f2, n0])
cat_tab = tabela(top_cat.set_index("categoria_en"),
                 ["pedidos_com_avaliacao", "kpi1_pct_avaliacao_ruim", "kpi3_pct_postagem_no_prazo", "pct_entrega_atrasada", "kpi2_pct_frete_sobre_produto", "excesso"],
                 ["Categoria", "Pedidos avaliados", "% aval. ruim", "% postagem no prazo", "% entrega atrasada", "% frete", "pedidos ruins acima da média"],
                 [n0, f2, f2, f2, f2, n0])
cross_tab = "\n".join(
    [f"| {p} | {e} | {n0(c(P25, p, e, 'pedidos_com_avaliacao'))} | {f2(c(P25, p, e))} |"
     for p in ["postagem no prazo", "postagem atrasada"] for e in ["entrega no prazo", "entrega atrasada"]])
q = qualidade.set_index("verificacao")["valor"]

relatorio = f"""# Relatório analítico — Olist Store | CP5 FIAP

> Gerado automaticamente pelo notebook `07_validacoes_exportacao.ipynb` a partir das tabelas calculadas. Nenhum número foi digitado à mão.

> **Premissa temporal (proxy acadêmica):** “2024” = pedidos com compra em **jan–ago/2017** e “2025” = **jan–ago/2018** no dataset original. O dataset **não** contém vendas reais de 2024/2025; os rótulos servem só para a comparação equivalente pedida na proposta. As datas originais não foram alteradas.

## 1. Resumo executivo
- **A avaliação ruim subiu** de {f2(r.loc[2017, 'kpi1_pct_avaliacao_ruim'])}% para {f2(r.loc[2018, 'kpi1_pct_avaliacao_ruim'])}% (+{f2(r.loc[2018, 'kpi1_pct_avaliacao_ruim'] - r.loc[2017, 'kpi1_pct_avaliacao_ruim'])} p.p.), enquanto a **entrega atrasada ao cliente** passou de {f2(r.loc[2017, 'pct_entrega_atrasada'])}% para {f2(r.loc[2018, 'pct_entrega_atrasada'])}%. A **postagem no prazo** pelos sellers ficou praticamente estável ({f2(r.loc[2017, 'kpi3_pct_postagem_no_prazo'])}% → {f2(r.loc[2018, 'kpi3_pct_postagem_no_prazo'])}%).
- **A hipótese recebe apoio parcial.** O atraso na entrega está fortemente associado à avaliação ruim ({f1(ent_atr.pct)}% de notas 1–2 com entrega atrasada vs {f1(ent_ok.pct)}% no prazo, proxy 2025). A postagem atrasada pelo seller também está associada, mas com intensidade bem menor ({f1(pos_atr.pct)}% vs {f1(pos_ok.pct)}%).
- **O atraso é mais logístico que de seller:** {f1(share_log)}% dos pedidos avaliados com entrega atrasada (proxy 2025) tiveram a **postagem no prazo**. Norte e Nordeste têm a maior taxa de entrega atrasada mesmo quando o seller posta no prazo, além do frete proporcional mais alto.
- **Tamanho da oportunidade (descritivo, não causal):** se os pedidos atrasados tivessem a mesma taxa de avaliação ruim dos pedidos no prazo, o KPI1 da proxy 2025 seria {f2(dec.loc[P25, 'kpi1_cenario_atrasados_com_taxa_do_prazo'])}% (−{f2(dec.loc[P25, 'reducao_pp_teorica'])} p.p.), abaixo da meta de 10%. O mesmo exercício com a postagem do seller levaria a {f2(decp.loc[P25, 'kpi1_cenario_postagem_atrasada_com_taxa_do_prazo'])}% (−{f2(decp.loc[P25, 'reducao_pp_teorica'])} p.p.).
- **Frete:** o KPI2 da proxy 2025 está em {f2(r.loc[2018, 'kpi2_pct_frete_sobre_produto'])}%, praticamente no limite de 17,1%. Não há folga para resolver o atraso pagando frete mais caro de forma generalizada.
- **Recomendação:** priorizar a **logística regional** (Nordeste/Norte e RJ) com SLA de prazo da transportadora e revisão do prazo estimado, e manter um programa **focado** nos sellers reincidentes (os 50 sellers com mais postagens atrasadas concentram {f1(conc.loc[50, 'pct_das_postagens_atrasadas'])}% dos atrasos de postagem).

## 2. Contexto e decisão
A Olist Store precisa decidir onde concentrar esforços para reduzir a insatisfação: **cobrar o prazo de postagem dos sellers** ou **renegociar/trocar transportadoras por região**. Meta: reduzir pedidos com avaliação 1–2 para **10% até dez/2027** (passo de **12% em 2026**), sem que o frete sobre o valor dos produtos passe de **17,1%**. Público: Diretor de Operações do Marketplace, Gestão de Sellers e Logística.

## 3. Fonte e recorte
- *Brazilian E-Commerce Public Dataset by Olist* (Kaggle), 8 tabelas relacionais usadas + `geolocation` (inventariada, não usada nos KPIs).
- Filtro: `order_status = delivered` e mês da compra entre janeiro e agosto de 2017 e 2018.
- Pedidos no recorte: **{n0(n_ano[2017])}** (2017) e **{n0(n_ano[2018])}** (2018), total **{n0(len(pedidos))}**. {'Os totais coincidem com a referência da proposta (21.998 / 52.783 / 74.781).' if bate_ref else 'ATENÇÃO: os totais diferem da referência da proposta (21.998 / 52.783); ver qualidade_dados.csv.'}

## 4. Definições dos KPIs
| KPI | Fórmula | Regras |
|---|---|---|
| KPI1 | pedidos entregues com nota 1–2 / pedidos entregues com avaliação válida × 100 | avaliação mais recente por `review_creation_date` (desempate: `review_answer_timestamp`, depois `review_id`); pedidos sem avaliação fora do denominador |
| KPI2 | Σ `freight_value` / Σ `price` dos itens dos pedidos entregues × 100 | somado no nível item; joins N:1 validados para não multiplicar frete |
| KPI3 | pedidos postados até o prazo / pedidos elegíveis × 100 | prazo = **menor** `shipping_limit_date` entre os itens; postagem exatamente no limite = no prazo; elegível = tem data de postagem e prazo |

Derivadas: entrega atrasada = dia de `order_delivered_customer_date` > dia de `order_estimated_delivery_date` (a data estimada não tem hora); região do cliente por mapa explícito das 27 UFs.

## 5. Qualidade e tratamento dos dados
- {n0(q['pedidos_com_mais_de_uma_avaliacao'])} pedidos da base têm mais de uma avaliação ({n0(q['pedidos_recorte_com_multiplas_avaliacoes'])} no recorte): usada a mais recente.
- Pedidos do recorte sem avaliação: {n0(q['pedidos_recorte_sem_avaliacao'])} ({f2(q['pedidos_recorte_sem_avaliacao'] / len(pedidos) * 100)}%), fora do denominador do KPI1.
- Excluídos do KPI3 por falta de data de postagem: {n0(q['kpi3_excluido_sem_data_postagem'])}; sem prazo limite: {n0(q['kpi3_excluido_sem_prazo_limite'])}.
- Excluídos da métrica de entrega atrasada por falta de data de entrega: {n0(q['atraso_entrega_excluido_sem_data_entrega'])}.
- Itens sem categoria: {n0(q['itens_sem_categoria'])} (mantidos como `sem_categoria`); itens com categoria sem tradução: {n0(q['itens_categoria_sem_traducao'])} (nome original mantido).
- {n0(q['postagem_antes_da_compra_no_recorte'])} pedidos têm data de postagem anterior à compra (inconsistência da base). Foram mantidos e sinalizados; não alteram a classificação “no prazo”.
- Itens com frete zero ({n0(q['itens_frete_zero'])}) são válidos (frete grátis) e foram mantidos. Não houve preço/frete negativo nem nota fora de 1–5.
- Nenhuma data, nota ou valor monetário foi imputado.

## 6. Resultados

### 6.1 KPIs por período (calculado × proposta)
| KPI | 2024 calculado | 2024 proposta | 2025 calculado | 2025 proposta | Meta 2026 | Meta 2027 |
|---|---:|---:|---:|---:|---:|---:|
{kpi_tab}

Os valores calculados coincidem com os da proposta quando arredondados a uma casa decimal. Isso confirma que as definições são as mesmas.

### 6.2 Evolução mensal (proxy 2025)
A piora da proxy 2025 está concentrada em **fevereiro e março/2018**: avaliação ruim de {f1(mens.loc[2, 'kpi1_pct_avaliacao_ruim'])}% e {f1(mens.loc[3, 'kpi1_pct_avaliacao_ruim'])}%, com entrega atrasada de {f1(mens.loc[2, 'pct_entrega_atrasada'])}% e {f1(mens.loc[3, 'pct_entrega_atrasada'])}%. De abril a agosto a avaliação ruim ficou entre {f1(mens.loc[4:8, 'kpi1_pct_avaliacao_ruim'].min())}% e {f1(mens.loc[4:8, 'kpi1_pct_avaliacao_ruim'].max())}%. A postagem no prazo nesses dois meses foi de {f1(mens.loc[2, 'kpi3_pct_postagem_no_prazo'])}% e {f1(mens.loc[3, 'kpi3_pct_postagem_no_prazo'])}%, sem queda equivalente. Ver `output/figuras/03_kpis_mensais.png`.

### 6.3 Atraso × avaliação ruim (proxy 2025)
| Postagem | Entrega | Pedidos avaliados | % avaliação ruim |
|---|---|---:|---:|
{cross_tab}

- Entrega atrasada vs no prazo: **{f1(ent_atr.pct)}% vs {f1(ent_ok.pct)}%** (+{f1(ent_atr.pct - ent_ok.pct)} p.p.; na proxy 2024: {f1(ent_atr24.pct)}% vs {f1(ent_ok24.pct)}%).
- Postagem atrasada vs no prazo: **{f1(pos_atr.pct)}% vs {f1(pos_ok.pct)}%** (+{f1(pos_atr.pct - pos_ok.pct)} p.p.).
- Postagem atrasada eleva a chance de entrega atrasada: {f1(pe_atr.pct)}% vs {f1(pe_ok.pct)}%.
- Com a entrega no prazo, a postagem atrasada eleva pouco a avaliação ruim ({f1(c(P25, 'postagem atrasada', 'entrega no prazo'))}% vs {f1(c(P25, 'postagem no prazo', 'entrega no prazo'))}%). Com a entrega atrasada, a taxa é alta nos dois casos.
- Todos os testes qui-quadrado deram p < 0,001. Eles indicam associação, **não** causalidade.

### 6.4 Regiões
{reg_tab}

{orig_tab}

O frete por região na proxy 2025 coincide com as referências da proposta (Norte 23,9%, Nordeste 22,7%, Sudeste 15,5%). No Norte e no Nordeste a entrega atrasa mais mesmo com a postagem no prazo, e o trecho postagem→entrega é mais longo. A taxa de postagem atrasada, por sua vez, é parecida entre as regiões.

### 6.5 UFs com mais pedidos ruins acima da taxa média (proxy 2025)
{uf_tab}

### 6.6 Categorias com mais pedidos ruins acima da taxa média (proxy 2025, ≥ {U.MIN_AMOSTRA} avaliações)
{cat_tab}

Categorias com menor postagem no prazo (≥ 200 pedidos): {", ".join(f"{i} ({f1(v)}%)" for i, v in low_post_cat.set_index("categoria_en")["kpi3_pct_postagem_no_prazo"].items())}.

### 6.7 Sellers
- Os 10, 50 e 100 sellers com mais postagens atrasadas concentram {f1(conc.loc[10, 'pct_das_postagens_atrasadas'])}%, {f1(conc.loc[50, 'pct_das_postagens_atrasadas'])}% e {f1(conc.loc[100, 'pct_das_postagens_atrasadas'])}% das postagens atrasadas. Eles respondem por {f1(conc.loc[10, 'pct_do_volume_pedido_seller'])}%, {f1(conc.loc[50, 'pct_do_volume_pedido_seller'])}% e {f1(conc.loc[100, 'pct_do_volume_pedido_seller'])}% do volume pedido–seller.
- O seller nº 1 do ranking tem {n0(top50c.iloc[0].pedido_seller_postagem_atrasada)} postagens atrasadas em {n0(top50c.iloc[0].pedido_seller_elegiveis_postagem)} ({f1(top50c.iloc[0].pct_postagem_atrasada)}%). Ver `top_50_sellers_atraso.csv` e `ranking_semanal_sellers_atraso.csv`.
- Os rankings são **diagnóstico retrospectivo** (2017–2018), não alertas atuais. As contagens são na granularidade pedido–seller e não somam pedidos únicos.

## 7. Conclusão: evidência, interpretação e recomendação
**Evidência observada**
1. Entrega atrasada ao cliente está associada a uma taxa de avaliação ruim ~{f1(ent_atr.pct / ent_ok.pct)}× maior.
2. {f1(share_log)}% das entregas atrasadas avaliadas (proxy 2025) tiveram postagem no prazo.
3. A postagem no prazo ficou estável entre os períodos, enquanto a entrega atrasada e a avaliação ruim cresceram, com pico em fev–mar/2018.
4. Norte/Nordeste combinam entrega mais lenta, mais atraso mesmo com a postagem no prazo e o frete proporcional mais alto. Pelo volume, o RJ é a UF com mais pedidos ruins acima da média.

**Interpretação**
- A hipótese recebe **apoio parcial**: o atraso na entrega é o fator mais associado à insatisfação, e a maior parte desse atraso **não** vem de postagem atrasada pelo seller. Os dados apontam para uma **combinação**, com peso maior no lado **logístico/regional**. A gestão de sellers é uma alavanca secundária, mas concentrada em poucos sellers.
- Não é possível separar com estes dados o efeito da transportadora, da distância e de uma promessa de prazo otimista: o dataset não identifica a transportadora.

**Recomendações priorizadas**
| # | Ação | Área | KPI de acompanhamento |
|---|---|---|---|
| 1 | SLA e renegociação com transportadoras nas rotas para Nordeste, Norte e RJ; revisão do prazo estimado nessas rotas | Logística | % entrega atrasada por região; KPI1 por região |
| 2 | Plano de capacidade para picos (o pico de fev–mar/2018 sugere gargalo sazonal) | Logística/Operações | % entrega atrasada mensal; KPI1 mensal |
| 3 | Programa de pontualidade para os sellers reincidentes (top 50 e ranking semanal), com alerta e penalidade progressiva | Gestão de Sellers | KPI3 (meta 95% em 2026 e 97% em 2027); % postagem atrasada do top 50 |
| 4 | Ações por categoria com postagem baixa (ex.: móveis de escritório) | Gestão de Sellers | KPI3 e KPI1 por categoria |
| 5 | Guarda-corpo de custo: toda mudança de transportadora deve ser avaliada pelo KPI2 geral (≤ 17,1%) e regional | Logística/Financeiro | KPI2 geral e por região |

**Como não mascarar a melhora com frete mais caro:** acompanhar o KPI1 e o KPI2 sempre lado a lado. Como o KPI2 já está em {f2(r.loc[2018, 'kpi2_pct_frete_sobre_produto'])}%, qualquer aumento de frete no Norte/Nordeste precisa ser compensado por ganho de eficiência em outras rotas. Na proxy 2025, os itens de pedidos entregues no prazo tiveram frete de {f2(fr_ent.loc[False])}% sobre o produto, contra {f2(fr_ent.loc[True])}% nos atrasados. Ou seja, pagar mais frete, por si só, não garante entrega no prazo.

## 8. Limitações e próximos passos
- Proxy temporal: 2017/2018 representam 2024/2025 apenas para fins acadêmicos.
- O dataset não informa a transportadora nem a data de postagem por seller: em pedidos com vários sellers, a data de postagem do pedido é atribuída a todos.
- Associação não é causalidade. Os próximos passos seriam um modelo multivariado (ex.: regressão logística com região, categoria, seller e mês) e testes A/B das ações.
- Grupos com menos de {U.MIN_AMOSTRA} avaliações estão sinalizados (`amostra_pequena`) e devem ser lidos com cautela.

## 9. Reprodução
Ver `README.md` na raiz. Saídas em `output/dashboard/` (CSVs) e `output/figuras/` (PNGs).
"""
(U.ROOT / "relatorio_analitico.md").write_text(relatorio, encoding="utf-8")
print("relatorio_analitico.md gerado com", len(relatorio.splitlines()), "linhas")

relatorio_analitico.md gerado com 144 linhas


## 10. Resultado das validações

In [12]:
v = pd.DataFrame(validacoes)
print(f"{int((v.etapa == '07_validacao').sum())} verificações registradas; falhas: {int((v.detalhe.str.startswith('FALHOU')).sum())}")
v[["verificacao", "valor", "detalhe"]]

113 verificações registradas; falhas: 0


,verificacao,valor,detalhe
0,pedidos: 1 linha por order_id,1,OK 74781 linhas
1,itens: 1 linha por order_id+order_item_id,1,OK
2,pedido_seller: 1 linha por order_id+seller_id,1,OK
3,pedido_seller identificado como tal (linhas > ...,1,OK 75845 linhas para 74781 pedidos únicos
4,itens: mesma quantidade que os itens originais...,1,OK
5,frete não multiplicado pelos joins (itens),1,OK 1712901.77
6,preço não multiplicado pelos joins (itens),1,OK
7,frete pedido_seller == frete itens,1,OK
8,frete pedidos == frete itens,1,OK
9,somente status delivered,1,OK
